# CelebA MobileNetV2 (Small Subset) – TensorFlow 2.x

This notebook uses the **CelebA-HQ small subset from Hugging Face**, avoiding Google Drive issues.

**Features:**
- Pre-split train/validation/test sets
- MobileNetV2 feature extractor (transfer learning)
- Data augmentation
- Fine-tuning and evaluation
- Visualization of predictions


In [ ]:
from datasets import load_dataset
import tensorflow as tf
import numpy as np

print("Importuri ok!")

: 

## Load CelebA Small Subset from Hugging Face

In [9]:
from datasets import load_dataset
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Load CelebA-HQ small (~2000 images)
dataset = load_dataset('ashraq/celeba-hq-small')

def to_tf_dataset(split):
    data = dataset[split]
    def gen():
        for x in data:
            yield x['image'], x['attributes']['Smiling']
    return tf.data.Dataset.from_generator(
        gen,
        output_signature=(tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
                          tf.TensorSpec(shape=(), dtype=tf.int64))
    )

ds_train = to_tf_dataset('train')
ds_val = to_tf_dataset('validation')
ds_test = to_tf_dataset('test')

print(ds_train, ds_val, ds_test)

TypeError: expected string or bytes-like object, got 'NoneType'

## Preprocess and Augment

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32

resize_and_rescale = tf.keras.Sequential([
    tf.keras.layers.Resizing(IMG_SIZE, IMG_SIZE),
    tf.keras.layers.Rescaling(1./255)
])

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

def prepare(ds, shuffle=False):
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.map(lambda x, y: (resize_and_rescale(x), tf.cast(y, tf.float32)))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = prepare(ds_train, shuffle=True).map(lambda x, y: (data_augmentation(x), y))
val_ds = prepare(ds_val)
test_ds = prepare(ds_test)

for img_batch, lbl_batch in train_ds.take(1):
    print('Image batch shape:', img_batch.shape)
    print('Label batch shape:', lbl_batch.shape)

NameError: name 'tf' is not defined

## Build and Train Model

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet')

base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='binary_crossentropy',
              metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_model.keras', save_best_only=True)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

NameError: name 'tf' is not defined

## Fine-Tune (Optional)

In [10]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

fine_tune_history = model.fit(train_ds, validation_data=val_ds, epochs=5)

NameError: name 'base_model' is not defined

## Evaluate and Visualize Predictions

In [12]:
test_loss, test_acc = model.evaluate(test_ds)
print(f'Test accuracy: {test_acc:.3f}')

for images, labels in test_ds.take(1):
    preds = model.predict(images)
    preds = (preds > 0.5).astype(int)
    plt.figure(figsize=(10, 10))
    for i in range(9):
        plt.subplot(3, 3, i+1)
        plt.imshow(images[i])
        plt.title(f'Pred: {preds[i][0]} | True: {int(labels[i])}')
        plt.axis('off')
    plt.show()

KeyboardInterrupt: 